# Lesson 04 - 工具使用設計模式

在本課中，您將學習使用 Microsoft Agent Framework (Python) 的 AI 代理的 **工具使用** 設計模式。我們將涵蓋：

- 使用 `@tool` 裝飾器和類型化參數定義函數工具
- 提供工具架構，讓模型知道每個工具的功能
- 透過 `approval_mode` 控制工具執行
- 透過 Pydantic 模型和 `response_format` 返回 **結構化輸出**

情境是一個 **旅遊訂票代理**，可以查詢目的地、檢查可用性及取得航班資訊。


## 設置


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器定義工具

`@tool` 裝飾器將普通的 Python 函數轉變為代理可以調用的工具。
要點：

- **文檔字串** 成為模型看到的工具描述。
- **類型註解**（包括帶有描述的 `Annotated`）定義工具結構。
- `approval_mode` 控制是否需要用戶在每次調用前批准。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 使用多個工具創建一個代理

將所有三個工具傳遞給客戶端，以便模型可以調用其需要的任何工具來回答用戶的問題。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## 使用工具的結構化輸出

通過將 `response_format` 設置為 Pydantic 模型，代理被強制返回一個類型良好的 JSON 對象，而不是自由格式的文本。當下游代碼需要以程式化方式消費結果時，這非常有用。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## 工具批准模式

`@tool` 上的 `approval_mode` 參數控制工具調用是否需要在執行前獲得人工批准：

| 模式 | 行為 |
|---|---|
| `"never_require"` | 工具自動執行 — 無需用戶確認。 |
| `"always_require"` | 每次調用必須經用戶批准後才執行。 |

對具有副作用的工具（例如預訂航班、信用卡扣款）使用 `"always_require"` 模式，以保持人工監管。


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

在本課程中，你學會了如何：

1. 使用帶有類型參數和作為工具架構的docstrings的`@tool`裝飾器**定義工具**。
2. **組合多個工具**，使代理能夠按順序調用它們以回答複雜查詢。
3. 通過傳遞Pydantic模型作為`response_format`，**返回結構化輸出**。
4. 通過`approval_mode`**控制工具審批**，以便在敏感操作中保持人工介入。

這些模式構成了構建可靠、生產就緒且能安全與外部系統交互的代理的基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件經由人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯。雖然我們致力於準確性，但請注意自動翻譯可能存在錯誤或不準確之處。原始文件以其母語版本為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯而引起的任何誤解或誤釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
